# Who is this player?

*Question, Intuition, Math, Code, Assumptions, How it breaks*

FBref does not give you a player ID. A row says
`("ESP-La Liga", "0405", "Barcelona", "Ronaldinho")` and that is everything you
get. There is no key anywhere linking that row to the same man at Paris
Saint-Germain two years earlier.

Every claim this project makes depends on getting past that. When I say
"Ronaldo's nineteen seasons across three countries are one career", that is a
sentence about identity long before it is a sentence about football.

## 1. Question

I have 67,704 rows keyed on nothing but a name and a club. How do I work out
which rows belong to the same person, without ever merging two people, and
without quietly deleting anybody?

## 2. Intuition

A name on its own is not enough, and this dataset proves it: there were two
players called Míchel at Rayo Vallecano in 2002-03.

A name plus a **birth year** is almost always enough. Birth year is the only
piece of independent evidence in the record. Everything else (club, position,
minutes) is a consequence of the career I am trying to reconstruct, so using any
of it would be circular.

That gives me two rules, and honestly they are the entire design:

1. **The birth year never relaxes.** A matching tier is allowed to loosen the
   name and nothing else.
2. **Ambiguity is a non-match.** Two candidates means no answer. A wrong ID is
   worse than a missing one, because a missing one gets reported and a wrong one
   silently welds two careers together and says nothing.

## 3. Math

There are two different things being built here, and confusing them causes
trouble, so let me separate them.

**The internal key** is a hash of the folded name and the birth year:

$$\text{player\_id} = \text{sha1}\big(\text{normalize}(\text{name}) \;|\; \text{born}\big)[:12]$$

It needs no external data, it never fails, and it is what actually groups rows
into careers.

**The Wikidata QID** is enrichment sitting on top. It is what lets me check the
ranking against award voting, and it is the part that can come back empty.

The ranking works fine without a QID. It does not work at all without a
`player_id`.

## 4. Code

### Folding a name

Two sources spell the same person differently. Accents, apostrophes and hyphens
all disagree. Here is the part that is easy to get wrong: **they do not all
disagree in the same direction.**

In [ ]:
import pandas as pd

from gambeta import tally, whois

for raw in ["Fàbregas", "M'Boma", "Jean-Pierre", "Nicolás Otamendi", "N'Golo Kanté"]:
    print(f"  {raw:<18} -> {whois.normalize(raw)}")

Look at rows two and three, because they pull in opposite directions.

FBref's `M'Boma` is Wikidata's `Mboma`, so the apostrophe has to **close up**.
FBref's `Jean-Pierre` is Wikidata's `Jean Pierre`, so the hyphen has to **open
out**.

Treat both as "punctuation" and collapse them the same way and you lose one of
the two, whichever way you pick. That is why there are two regexes in there and
not one.

### The key that makes a career

The identifier has to do two opposite jobs at the same time. Hold one person
together across every club and every season, and keep two people apart when they
happen to share a name.

In [ ]:
one_man = [("Cristiano Ronaldo", 1985.0), ("Cristiano Ronaldo", 1985.0)]
namesakes = [("Michel", 1975.0), ("Michel", 1977.0)]

print("same player, different clubs and seasons:")
for name, born in one_man:
    print(f"  {name} ({born:.0f}) -> {whois.player_id(name, born)}")

print("\ntwo players, same name, same club, same season:")
for name, born in namesakes:
    print(f"  {name} ({born:.0f}) -> {whois.player_id(name, born)}")

Those are the two Míchels from the validation chapter, and here they come out
correctly separated, because birth year is in the key.

### Matching in tiers

An exact join gets most of the way. What it misses is rarely a mangled name. It
is almost always a name written at a **different length**. When I measured it on
the real data, 90% of the unmatched players had a name that was absent under its
exact spelling and sitting right there under another one.

In [ ]:
crosswalk = pd.DataFrame(
    {
        "qid": ["Q1", "Q2", "Q3", "Q4", "Q5", "Q6"],
        "label": [
            "Cesc Fàbregas",  # exact, once folded
            "Yakubu Aiyegbeni",  # FBref writes just "Yakubu"
            "Matthew Upson",  # FBref writes "Matt Upson"
            "Iván Kaviedes Toaquiza",
            "Ali Hassan",  # two people, one birth year
            "Ali Hussein",
        ],
        "birth_year": [1987, 1982, 1979, 1977, 1990, 1990],
    }
)

fbref = pd.DataFrame(
    {
        "player": ["Cesc Fabregas", "Yakubu", "Matt Upson", "Iván Kaviedes", "Ali H"],
        "born": [1987.0, 1982.0, 1979.0, 1977.0, 1990.0],
    }
)

labelled, unresolved = whois.resolve(fbref, crosswalk)
labelled[["player", "born", "qid"]]

Four out of five matched, and each one took a different route: an accent fold, a
name contained inside a longer one, a shortened forename sharing an initial, and
a dropped second surname.

The fifth is the one I care about. `Ali H` could be either 1990 candidate, so it
gets **nothing** back, and it lands in the report rather than the bin.

In [ ]:
print(f"rows in : {len(fbref)}")
print(f"rows out: {len(labelled)}   <- resolution never changes the row count")
print("\nunresolved and reported, not dropped:")
unresolved[["player", "born", "player_id"]]

That length check is not decoration. Dropping an unmatched row raises no error
and passes every test I have. It just deletes a career. On the real data the same
report gets written to `vault/clean/unresolved.csv`, and there are currently
1,240 players in it.

### Collapsing a transfer season

FBref lists a mid-season transfer once per club. Two rows, one season, one
player. They have to become one row, and here is the catch: **not everything
adds up.**

In [ ]:
half = dict(
    qid="Q1",
    league="ENG-Premier League",
    player="A Forward",
    born=1979.0,
    nation="uy",
    pos="FW",
    season="0405",
    player_id="abc123",
)

mid_season = pd.DataFrame(
    [
        {**half, "team": "First Club", "minutes": 900, "mp": 12, "goals": 3, "min_pct": 26.3},
        {**half, "team": "Second Club", "minutes": 1800, "mp": 24, "goals": 8, "min_pct": 52.6},
    ]
)
for column in [
    "assists",
    "npg",
    "yellow",
    "red",
    "starts",
    "sot",
    "complete",
    "subs",
    "second_yellow",
    "fouls",
]:
    mid_season[column] = 0.0

collapsed = tally.collapse_transfers(mid_season)
collapsed[["player", "season", "teams", "minutes", "mp", "goals", "min_pct"]]

Minutes, appearances and goals add up. **The share of team minutes does not.**

In [ ]:
naive = mid_season["min_pct"].sum()
correct = collapsed["min_pct"].item()

print(f"summed  : {naive:.1f}% of his team's minutes")
print(f"correct : {correct:.1f}%")
print("\nHe was at two clubs. Adding the two shares asks what percentage of a")
print("season he played, and answers with a number that has no denominator.")

26.3% of one club's minutes plus 52.6% of another's is not 78.9% of anything at
all. What has to happen is that each share gets converted back into the minutes
it came from, those get summed, and then the real share is taken against the
total.

I know this because summing it directly gave one player **296%** of his team's
minutes, and the availability requirement read that as a virtue. That is the
second bug from the validation chapter, and this function is where it lived.

### What the published data looks like once that is done

In [ ]:
seasons = pd.read_parquet("../data/sample/player_season_scored.parquet")
moved = seasons[seasons["teams"].str.contains(",")]

print(f"{len(moved):,} of {len(seasons):,} player-seasons collapse a mid-season move")
print(f"max share of team minutes now: {seasons['availability'].max():.1%}\n")
shown = ["player", "season", "teams", "minutes", "mp", "availability"]
moved.nlargest(4, "minutes")[shown].round(3)

## 5. Assumptions

1. **Birth year is recorded and correct.** It is missing for 36 rows out of
   67,704, and I assume it is accurate everywhere else. One wrong year splits a
   single career into two, and nothing would tell me.
2. **One name plus one birth year is one person.** Two footballers born in the
   same year with the same folded name get silently merged. The tiers refuse
   ambiguity they can *see*, but identical keys are invisible to them.
3. **Wikidata is a reasonable authority.** Not for the ranking, for the awards
   check. A player who is absent from Wikidata still gets ranked.

## 6. How it breaks

**My diagnosis was wrong for weeks, and that is the part worth telling.**

The match rate sat at 83.9%. It looked exactly like a name-normalisation problem.
Accents, apostrophes, all the usual suspects. So I spent real effort folding
names harder, and it barely moved.

It was a **query** problem. The crosswalk was selecting people from Wikidata by
their FBref-ID property, which looked authoritative and which the matcher never
joined on anyway. That one clause threw away 56% of the candidate pool before
matching had even started. Anchoring on occupation instead, plus the tiers above,
took it to **93.9%**.

Weeks on the wrong layer, because the symptom (names not matching) pointed
straight at the wrong cause (name normalisation).

### What is left is genuinely hard

In [ ]:
hard = [("Serhiy", "Serhii"), ("Alexander", "Aliaksandr"), ("Luka Modrić", "Лука Модрич")]

for fbref_name, wikidata_label in hard:
    a, b = whois.normalize(fbref_name), whois.normalize(wikidata_label)
    verdict = "match" if a == b else "MISS"
    print(f"  {fbref_name:<12} -> {a:<12} | {wikidata_label:<12} -> {b:<12} {verdict}")

Three failure modes, and folding harder fixes none of them:

- **Transliteration.** `Serhiy` and `Serhii` are the same Ukrainian name through
  two conventions. Catching them needs a phonetic or edit-distance tier, and
  every loosening risks a wrong match, which rule 2 says is worse than a miss.
- **Non-Latin labels.** Modrić's Cyrillic label does not merely fail to match. It
  folds to an **empty string**, because the normaliser keeps only Latin letters
  and digits. He resolves through other rows in the crosswalk. The general case
  does not.
- **Absence.** Some players are simply not in Wikidata. No amount of matching
  invents them.

So I stopped, at diminishing returns. The better instrument turned out to be the
awards check, because it **names** the failures it finds. It caught Cristiano
Ronaldo being unresolved within minutes of first running, which no aggregate
match-rate percentage was ever going to do.

A number going from 83.9% to 93.9% tells you almost nothing about whether the
10% you still do not have contains anybody who matters.